#### invert_data.ipynb
This notebook imports continental flooding map data (provided in a binary state: 0=submerged/1=exposed) referenced to h3 grids, and uses logistic regression to perform time-dependent TPW inversions on them (by looking for correlations with a $Y_{21}$ pattern on rotation). Statistical significance is assessed with reference to the results of the permutation_tests.ipynb notebook output.

The notebook outputs time-dependent results of the inversions as well as a summary of the best-fitting inversions as a time-series.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import methods as m
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
### user-specified inputs ###
eps = 2.0        # cluster length-scale used by permutation results (degrees)
start = 0        # start time (in Ma) = youngest reconstruction step
stop = 320       # stop time (in Ma) = oldest reconstruction step
step = 10        # temporal step size (in Ma)

apply_tectonic_filter = False # if True: observations from newly formed passive margins and active foreland basins are removed
apply_glacial_filter = False # if True: observations from Antarctica since 30 Ma are removed

datdir = f'data/inversions'              # directory to output inversion data
figdir = f'figs/inversions/eps_{eps}'    # directory to output inversion figures
permdir = f'data/permutations/eps_{eps}' # location of permutation results directory
if apply_tectonic_filter or apply_glacial_filter:
    datdir += '/filtered'; figdir += '/filtered'; permdir += '/filtered' # filtered variants default to a 'filtered' subdirectory
os.makedirs(datdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

In [ ]:
# setup reference grids for plotting inversion results
refpts = pd.read_csv(f'obs_grids/{start}_Ma.csv')
rlats, rlons = m.equisphere(len(refpts))
revals, revecs = m.eigendecomp(rlats, rlons)
ref_power = np.sum(revals)

# cycle over time-steps
best_fits = []
sense = []
for t in range(start, stop, step):
    print (f'processing interval {t}-{t+step} Ma')
    t0 = pd.read_csv(f'obs_grids/{t}_Ma.csv')                # read in observation grids
    t1 = pd.read_csv(f'obs_grids/{t+step}_Ma.csv')
    merged = pd.merge(t0, t1, on='name', suffixes=('_t0', '_t1'))  # merge dataframes
    merged = merged.dropna(subset=['state_t0', 'state_t1'])        # drop NaNs
    if apply_tectonic_filter:
        merged = merged[~(merged['in_margin_t0'] | merged['in_margin_t1'] | merged['in_basin_t0']  | merged['in_basin_t1'])]  # get rid of any 'True' values
    if apply_glacial_filter and t <= 30:
        merged = merged[merged['lat_t0'] >= -60]                   # remove all points above -60 S latitude since 30 Ma
    merged['diff'] =  merged['state_t1'] - merged['state_t0']      # find points where there is flux: -1 = regression / 1 = transgression

    fluxpts = merged[merged['diff'] != 0]    # isolate points where there is flux
    lats = fluxpts['lat_t0'].to_numpy()
    lons = fluxpts['lon_t0'].to_numpy()
    flux = fluxpts['diff'].to_numpy()
    y = (flux == 1).astype(int)              # convert -1/1 range to 0/1 for logistic regression

    df = m.invert_lam_logistic(lats, lons, y)         # run inversion
    df.to_csv(f'{datdir}/{t}_Ma.csv', index=False)
    mask = df['beta'] > 0
    best_idx = df.loc[mask, 'llr'].idxmax()
    
    perm_results = pd.read_csv(f'{permdir}/{t}_Ma.csv')  # import corresponding permutation results
    p95 = np.percentile(perm_results['llr'], 95)             # sort and get 95 percentile
    best_fits.append([t, df.loc[best_idx, 'lambda'], df.loc[best_idx, 'llr'], df.loc[best_idx, 'alpha'], df.loc[best_idx, 'beta'], 
                      df.loc[best_idx, 'p-rsquared'], p95])

    if df.loc[best_idx, 'llr'] >= p95:
        significance = 'significant ($p < 0.05$)'
    else:
        significance = 'not significant ($p ≥ 0.05$)'
    
    # calculate resolving power and anisotropy
    evals, evecs = m.eigendecomp(t0['lat'], t0['lon'])
    power = np.sum(evals)
    norm_power = power/ref_power
    aniso = evals[0]/evals[1]
    sense.append([t, norm_power, aniso])

    s = m.Y21(df.loc[best_idx, 'lambda'], lats, lons)  # get predictor values for best-fitting model

    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(2, 6, height_ratios=[1, 1], hspace=0.4, wspace=-0.3)
    ax0 = fig.add_subplot(gs[0, 0:4])
    m.plot_inversion(df, p95, ax=ax0)  # compare observed vs. permutation best-fits
    ax1 = fig.add_subplot(gs[1, 0:3])  
    m.plot_log_reg(s, y, ax=ax1)
    ax2 = fig.add_subplot(gs[1, 3:6])
    m.plot_resolving_pwr(evals, evecs, ref_power, ax=ax2)
    fig.suptitle(f'Inversion results {t+step}–{t} Ma: {significance}', fontsize=14, y=1.00)
    fig.subplots_adjust(left=0.24)
    
    plt.savefig(f'{figdir}/{t}_Ma.pdf')
    plt.savefig(f'{figdir}/{t}_Ma.png')
    plt.close()

In [ ]:
# now plot the best fits through time
df_best = pd.DataFrame(best_fits, columns=['time', 'lambda', 'llr', 'alpha', 'beta', 'p-rsquared', 'p95'])
df_best['time'] = df_best['time'] + 5  # adjust for plotting at mid-points
df_sense = pd.DataFrame(sense, columns=['time', 'norm_power', 'aniso'])
df_sense['time'] = df_sense['time'] + 5

mask = df_best['llr'] > df_best['p95']   # boolean mask for significant results
intervals = []
for i in df_best['time'][mask]:
    intervals.append((i-step/2, i+step/2))

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [0.75, 0.75, 0.75, 1, 0.75]})
ax1.plot(df_best['time'], df_best['p-rsquared'], color='#333333', label='pseudo-$R^2$', zorder=0)
ax1.scatter(df_best['time'][mask], df_best['p-rsquared'][mask], facecolors='#333333', edgecolors='black', s=40, zorder=1)
for start, end in intervals: ax1.axvspan(start, end, color='tab:orange', linewidth=0, alpha=0.15, zorder=-1)
ax1.set_ylabel('Pseudo-$R^2$')
ax1.legend(loc='upper left')
ax1.grid(axis='x', linestyle='--', alpha=0.5)

ax2.plot(df_best['time'], df_best['llr'], color='tab:blue', label='best-fitting LLR', zorder=1)
#ax2.plot(df_best['time'], df_best['p95'], linestyle=':', color='tab:blue', label='$95^{th}$ pctl., '+ f'$\epsilon={eps}°$')
ax2.fill_between(df_best['time'], 0, df_best['p95'], color='#c7dded', linewidth=0, alpha=1, label='significance threshold', zorder=0)
ax2.scatter(df_best['time'][mask], df_best['llr'][mask], facecolors='tab:blue', edgecolors='black', s=40, zorder=1)
for start, end in intervals: ax2.axvspan(start, end, color='tab:orange', linewidth=0, alpha=0.15, zorder=-1)
ax2.set_ylabel('LLR')
ax2.legend(loc='upper left')
ax2.grid(axis='x', linestyle='--', alpha=0.5)

ax3.plot(df_best['time'], df_best['alpha'], color='tab:purple', linestyle=':', zorder=0, label=r'$\alpha$')
ax3.plot(df_best['time'], df_best['beta'], color='tab:red', linestyle='--', zorder=0, label=r'$\beta$')
ax3.scatter(df_best['time'][mask], df_best['beta'][mask], facecolors='tab:red', edgecolor='black', s=40, zorder=1)
ax3.scatter(df_best['time'][mask], df_best['alpha'][mask], facecolors='tab:purple', edgecolor='black', s=40, zorder=1)
for start, end in intervals: ax3.axvspan(start, end, color='tab:orange', linewidth=0, alpha=0.15, zorder=-1)
ax3.axhline(y=0, color='k', linewidth=0.75, zorder=0)
ax3.set_ylabel('Coeff. mag.')
ax3.legend(loc='upper left')
ax3.grid(axis='x', linestyle='--', alpha=0.5)

acw = df_best['lambda'] < 180  
cw = ~acw                     # if lambda is above 180 will flip and make ccw
ax4.scatter(df_best.loc[acw & ~mask, 'time'], df_best.loc[acw & ~mask, 'lambda'], color='tab:orange', edgecolor='none', s=20, alpha=0.4, label='acw rot')
ax4.scatter(df_best.loc[cw & ~mask, 'time'], df_best.loc[cw & ~mask, 'lambda'] - 180, color='tab:blue', edgecolor='none', s=20, alpha=0.4, label='cw rot')
ax4.scatter(df_best.loc[acw & mask, 'time'], df_best.loc[acw & mask, 'lambda'], color='tab:orange', edgecolors='black', s=50)
ax4.scatter(df_best.loc[cw & mask, 'time'], df_best.loc[cw & mask, 'lambda'] - 180, color='tab:blue', edgecolors='black', s=50)
for start, end in intervals: ax4.axvspan(start, end, color='tab:orange', linewidth=0, alpha=0.15, zorder=-1)
ax4.set_ylabel('Longitude (°E)')
ax4.legend(loc='upper left')
ax4.grid(axis='x', linestyle='--', alpha=0.5)

ax5.plot(df_sense['time'], df_sense['norm_power'], color='k', label='resolving power')
for start, end in intervals: ax5.axvspan(start, end, color='tab:orange', linewidth=0, alpha=0.15, zorder=-1)
ax5.set_xlabel('Time (Ma)')
ax5.set_ylabel('Rel. res. pwr.')
ax5.grid(axis='x', linestyle='--', alpha=0.5)

ax5b = ax5.twinx()
ax5b.plot(df_sense['time'], df_sense['aniso'], color='tab:purple', linestyle=':', label='anisotropy')
ax5b.set_ylabel('Anisotropy', color='tab:purple')
ax5b.tick_params(axis='y', labelcolor='tab:purple')
handles1, labels1 = ax5.get_legend_handles_labels()
handles2, labels2 = ax5b.get_legend_handles_labels()
ax5.legend(handles=handles1 + handles2, labels=labels1 + labels2, loc='lower left')

for ax in [ax1, ax2, ax3, ax4, ax5, ax5b]:
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.xaxis.label.set_size(9)
    ax.yaxis.label.set_size(9)

plt.tight_layout(h_pad=0.1)
plt.savefig(f'{figdir}/summary.pdf', bbox_inches='tight')
plt.savefig(f'{figdir}/summary.png', bbox_inches='tight')
#plt.show()